In [1]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

In [2]:
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

In [14]:
# test_imports.py
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

# Test 1: Check if functions are imported correctly
print("Function check:")
print("subPath exists:", subPath is not None)
print("participantsInfoPath exists:", participantsInfoPath is not None)
print("processSubPSDs exists:", processSubPSDs is not None)
print("processSub exists:", processSub is not None)

# Test 2: Check if path functions return expected values
print("\nPath check:")
try:
    participants_path = participantsInfoPath()
    print("Participants info path:", participants_path)
    print("Path exists:", os.path.exists(participants_path))
except Exception as e:
    print("Error getting participants path:", str(e))

# Test 3: Check if subPath works for a sample subject
print("\nsubPath check:")
try:
    subject_id = "001"  # Change to a subject ID you know exists
    path = subPath(subject_id, derivatives=True)
    print(f"Path for subject {subject_id}:", path)
    print("Path exists:", os.path.exists(path))
except Exception as e:
    print(f"Error getting path for subject {subject_id}:", str(e))

# Only run this if you're confident the above tests passed
# as this will attempt to actually load data
print("\nMini processSub check:")
try:
    import time
    start = time.time()
    subject_id = "001"  # Use a known subject ID
    print(f"Attempting to get first epoch for {subject_id}...")
    epochs = processSub(subject_id)
    print(f"Got {len(epochs)} epochs in {time.time() - start:.2f} seconds")
    print("First epoch shape:", epochs[0].get_data().shape)
except Exception as e:
    print(f"Error processing subject {subject_id}:", str(e))
print("check end")

Function check:
subPath exists: True
participantsInfoPath exists: True
processSubPSDs exists: True
processSub exists: True

Path check:
Participants info path: /Users/user/eeg-ds004504/ds004504/participants.tsv
Path exists: True

subPath check:
subPath 001
Path handed: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Path for subject 001: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Path exists: True

Mini processSub check:
Attempting to get first epoch for 001...
processSub 001
subPath 001
Path handed: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 398 epochs in 0.66 seconds
First epoch shape: (1, 19, 1501)
check end


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from schema_definition import get_subject_schema, get_feature_schema
from feature_extraction import processEpoch, processSub
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

# Getting all subjects information

In [16]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame


def load_subjects_df(spark: SparkSession, participants_path: str) -> DataFrame:
    """
    Reads participants.tsv and returns a Spark DataFrame
    with columns SubjectID and Group for groups A, C, and F.

    Parameters:
        spark (SparkSession): Active Spark session
        participants_path (str): Path to the participants.tsv file

    Returns:
        Spark DataFrame with SubjectID and Group columns
    """
    participantsInfo = pd.read_table(participants_path)

    records = []
    for group_code in ["A", "C", "F"]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group_code]["participant_id"].tolist()
        for sub in group_subjects:
            records.append((sub, group_code))
    return spark.createDataFrame(records, schema=get_subject_schema())


In [17]:
spark = SparkSession.builder.appName("MyApp").getOrCreate()

# subject_df = load_subjects_df(spark, "../ds004504/participants.tsv")
subject_df = load_subjects_df(spark, participantsInfoPath())

# subjects_df.show()
# subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

In [18]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame


def load_subjects_df(spark: SparkSession, participants_path: str) -> DataFrame:
    """
    Reads participants.tsv and returns a Spark DataFrame
    with columns SubjectID and Group for groups A, C, and F.

    Parameters:
        spark (SparkSession): Active Spark session
        participants_path (str): Path to the participants.tsv file

    Returns:
        Spark DataFrame with SubjectID and Group columns
    """
    participantsInfo = pd.read_table(participants_path)

    records = []
    for group_code in ["A", "C", "F"]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group_code]["participant_id"].tolist()
        for sub in group_subjects:
            records.append((sub, group_code))
    return spark.createDataFrame(records, schema=get_subject_schema())


In [19]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

# Now create a new SparkSession with local binding address
from pyspark.sql import SparkSession
import os

# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()

print("New Spark session created successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully
New Spark session created successfully


In [20]:
spark = SparkSession.builder.appName("MyApp").getOrCreate()

subject_df = load_subjects_df(spark, "../ds004504/participants.tsv")

# subjects_df.show()
# subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

25/03/31 17:02:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [21]:

subject_df = load_subjects_df(spark, "../ds004504/participants.tsv")

# subjects_df.show()
# subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

In [22]:
subject_df.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- Group: string (nullable = false)



In [23]:
@pandas_udf(get_feature_schema(), PandasUDFType.GROUPED_MAP)
def extract_features_udtf(pdf):
    from feature_extraction import processEpoch, processSub
    from schema_definition import get_feature_schema, get_subject_schema
    import mne
    print("function ran")
    rows = []
    for _, row in pdf.iterrows():
        subject_id = row["SubjectID"]
        try:
            print(f"Processing subject {subject_id}")
            epochs = processSub(subject_id, derivatives=False)
            print(f"Got {len(epochs)} epochs for {subject_id}")
            print(type(epochs)) 
            # for i, epoch in enumerate(epochs):
            for i in range(len(epochs)):
                epoch = epochs[i]
                if i < 2:  # Just print info for the first 2 epochs to avoid spam
                    # print(f"Epoch {i} shape: {epoch.to_data_frame().shape()}")
                    print(f"Epoch {i}")
                    print(type(epoch))
                    print(type(epochs[i]))
                epoch_id = f"ep-{i}"
                features = processEpoch(epoch)
                
                if i < 2:  # Debug output
                    print(f"Epoch {i} features count: {len(features) if features else 0}")
                    if features and len(features) > 0:
                        pass
                        # print(f"First feature sample: {next(iter(features))}")
                
                for item in features:
                    # Check the structure of each item
                    # print(f"item {item}")
                    electrode_band_key, stats_value = item
                    electrode, band = electrode_band_key
                    # print(f"Adding: {subject_id}, {epoch_id}, {band}, {electrode}, stats: {stats_value}")
                    
                    # Add to results - adjust this based on actual structure 
                    # TODO : **Error here ! have to line it up! :)
                    try:
                        rows.append((subject_id, epoch_id, band, electrode, *stats_value))
                    except Exception as e:
                        print(f"Error appending row: {e}, stats_value: {stats_value}")
            
            print(f"Total rows collected: {len(rows)}")
            
        except Exception as e:
            print(f"Error processing {subject_id}: {e}")
            import traceback
            traceback.print_exc()
            
    # Print final row count before returning
    print(f"Returning DataFrame with {len(rows)} rows")
    
    # Check if we have column names from schema
    schema_fields = get_feature_schema()
    column_names = [f.name for f in schema_fields]
    print(f"Column names from schema: {column_names}")
    #error after here 
    import pandas as pd
    result_df = pd.DataFrame(rows, columns=column_names)
    print(f"Result DataFrame shape: {result_df.shape}")
    return result_df

In [24]:
print(processSub('sub-001', derivatives=False))

processSub sub-001
subPath sub-001
Path handed: /Users/user/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
<Epochs | 398 events (all good), 0 – 3 s (baseline off), ~86.6 MiB, data loaded,
 '1': 398>


In [25]:

result = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001") & (subject_df.Group == "A"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

result.show()

/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
25/03/31 17:02:56 ERROR Executor: Exception in task 0.0 in stage 2.0 (TID 12) 1]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/var/folders/xj/mwp5pjc12yd_2c1tnd3t29s00000gp/T/ipykernel_14892/2441551794.py", line 3, in extract_features_udtf
ModuleNotFoundError: No module named 'feature_extraction'

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.PythonArrowOutput$$anon$1.read(PythonArrowOutput.scala:118)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at sc

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/var/folders/xj/mwp5pjc12yd_2c1tnd3t29s00000gp/T/ipykernel_14892/2441551794.py", line 3, in extract_features_udtf
ModuleNotFoundError: No module named 'feature_extraction'
